In [ ]:
from matplotlib import pyplot as plt
from scipy.stats import norm
import numpy as np 
import csv
import math
import sys
import VSPFunctions as vsp
import pandas as pd
import seaborn as sns
from scipy.optimize import curve_fit
from scipy.stats import lognorm
import os
from astropy.table import Table
from astropy.io import fits

directory = "/users/cdcook/VSP/graphics/"

sns.set_theme(style="darkgrid")

In [ ]:
file_path = '/lustre/work/client/users/cdcook/fits_structs/000415_xtetrans_1a_match.fit'

def find_limiting_magnitude(data):
    if 'M' in data.columns.names and 'JD' in data.columns.names:
        magnitudes = data['M']
        julian_dates = data['JD']
        num_exposures = magnitudes.shape[2]  # Number of exposures

        # Debugging statement to check the shape of the Julian Dates array
        print("Shape of Julian Dates array:", julian_dates.shape)

        results = []
        for exposure_index in range(num_exposures):
            # Get magnitudes for the current exposure
            magnitudes_exposure = magnitudes[0, :, exposure_index]

            # Get the Julian Date for the current exposure
            julian_date = julian_dates[0, exposure_index] if julian_dates.ndim > 1 else julian_dates[exposure_index]

            # Apply condition to ignore magnitudes < 5 or > 25
            valid_magnitudes = magnitudes_exposure[(magnitudes_exposure >= 5) & (magnitudes_exposure <= 25)]

            # Find the maximum magnitude within the valid range
            if len(valid_magnitudes) > 0:
                limiting_mag = np.max(valid_magnitudes)
            else:
                limiting_mag = np.nan  # Handle case where all valid magnitudes are filtered out

            results.append((julian_date, limiting_mag))

        # Remove the last item from the list
        results.pop()

        julian_dates, limiting_mags = zip(*results)
        
        return julian_dates, limiting_mags
    else:
        print("M or JD column not found in the data.")
        return [], []

# Open the FITS file
with fits.open(file_path) as hdul:
    # Access the data (assuming data is in a table in the first extension HDU)
    if len(hdul) > 1:
        data_hdu = hdul[1]
        data = data_hdu.data

        # Find the limiting magnitude and Julian Date for each exposure
        julian_dates, limiting_mags = find_limiting_magnitude(data)

        # Print or use the list of results
        for i, (jd, mag) in enumerate(zip(julian_dates, limiting_mags)):
            print(f"Exposure {i+1}: Julian Date = {jd}, Limiting Magnitude = {mag:.2f}")
    else:
        print("No data found in the first extension HDU.")

In [ ]:
# Function to check if a specific flag is true
def flag_is_true(row, flag):
    return row.get(flag, False)

def KronCut(kronBand, kronDist, dataDF):
    if (kronBand == 'g'):
        kronName = 'gMeanKronMag'
        psfName = 'gMeanPSFMag'
        kronCutName = 'gKron'
    if (kronBand == 'r'):
        kronName = 'rMeanKronMag'
        psfName = 'rMeanPSFMag'
        kronCutName = 'rKron'
    if (kronBand == 'i'):
        kronName = 'iMeanKronMag'
        psfName = 'iMeanPSFMag'
        kronCutName = 'iKron'
    if (kronBand == 'z'):
        kronName = 'zMeanKronMag'
        psfName = 'zMeanPSFMag'
        kronCutName = 'zKron'
    if (kronBand == 'y'):
        kronName = 'yMeanKronMag'
        psfName = 'yMeanPSFMag'
        kronCutName = 'yKron'
        
    # Condition 1: psfName - kronName should be less than kronDist
    condition1 = abs(dataDF[psfName] - dataDF[kronName]) < kronDist
    # Apply both conditions to the DataFrame
    dataDF = dataDF[condition1]
    
    return dataDF, kronName, psfName, kronCutName

def BitFlagCut(bitFlags, dataDF):
    dataDF = dataDF[dataDF['Flags'] == bitFlags]
    return dataDF

def ColorCut(dataDF, sub1, sub2):
    if (sub1 == 'gr'):
        band1a = 'gMeanPSFMag'
        band1b = 'rMeanPSFMag'
        band1name = "g-r"
        
    elif (sub1 == 'gi'):
        band1a = 'gMeanPSFMag'
        band1b = 'iMeanPSFMag'
        band1name = "g-i"
        
    elif (sub1 == 'ri'):
        band1a = 'rMeanPSFMag'
        band1b = 'iMeanPSFMag'
        band1name = "r-i"
        
    if (sub2 == 'gr'):
        band2a = 'gMeanPSFMag'
        band2b = 'rMeanPSFMag'
        band2name = "g-r"
    
    elif (sub2 == 'gi'):
        band2a = 'gMeanPSFMag'
        band2b = 'iMeanPSFMag'
        band2name = "g-i"
        
    elif (sub2 == 'ri'):
        band2a = 'rMeanPSFMag'
        band2b = 'iMeanPSFMag'
        band2name = "r-i"
        
    x = []
    y = []
    for index, row in dataDF.iterrows():
        x.append(row[band1a] - row[band1b])
        y.append(row[band2a] - row[band2b])

    dataDF.insert(0, 'color1', x)
    dataDF.insert(1, 'color2', y)
    return dataDF, band1a, band1b, band2a, band2b, band1name, band2name

In [ ]:
slopeList = []
ABoffsetList = []
countsList = []
exposures = []

kronCutTF = True

In [ ]:
for n in range(55):
    exposure = n
    print(n)
    title = '/lustre/work/client/users/cdcook/VSPData/meanFields/meanXtetrans_1a0415Files/Pan000415xtetrans_1a_exp' + str(exposure) + '.csv'
    panDF = pd.read_csv(title)
    panDF = panDF.drop_duplicates(subset=['RA', 'Dec'])
    panDF = panDF[(panDF['Mag'] >= 5) & (panDF['Mag'] <= 25)]
    panDF = panDF[panDF['gMeanPSFMag'] != -999]
    panDF = panDF[panDF['rMeanPSFMag'] != -999]
    panDF = panDF[panDF['iMeanPSFMag'] != -999]
    panDF = panDF[panDF['zMeanPSFMag'] != -999]
    panDF = panDF[panDF['yMeanPSFMag'] != -999]
    panDF = panDF[panDF['gMeanKronMag'] != -999]
    panDF = panDF[panDF['rMeanKronMag'] != -999]
    panDF = panDF[panDF['iMeanKronMag'] != -999]
    panDF = panDF[panDF['zMeanKronMag'] != -999]
    panDF = panDF[panDF['yMeanKronMag'] != -999]
    
    # Define a function to calculate the flux
    def calculate_flux(magnitude):
        return np.power(10, (magnitude + 48.6) / -2.5)
    
    # Calculate flux for each band
    panDF['gflux'] = panDF['gMeanPSFMag'].apply(calculate_flux)
    panDF['rflux'] = panDF['rMeanPSFMag'].apply(calculate_flux)
    panDF['iflux'] = panDF['iMeanPSFMag'].apply(calculate_flux)
    panDF['zflux'] = panDF['zMeanPSFMag'].apply(calculate_flux)
    panDF['yflux'] = panDF['yMeanPSFMag'].apply(calculate_flux)
    
    # Calculate total flux
    panDF['totalFlux'] = (
        panDF['gflux'] * 0.1212 + 
        panDF['rflux'] * 0.1463 + 
        panDF['iflux'] * 0.1435 + 
        panDF['zflux'] * 0.098 + 
        panDF['yflux'] * 0.0393
    ) / 0.5483
    
    # Calculate logpart
    panDF['logpart'] = np.log10(panDF['totalFlux'] / 3631e-23)
    
    # Calculate pseudoBoloMag
    panDF['pseudoBoloMag'] = -2.5 * panDF['logpart']
    
    flux_columns = ['gflux', 'rflux', 'iflux', 'zflux', 'yflux']
    panDF.drop(columns=flux_columns, inplace=True)
    
    panDF['Difference'] = panDF['pseudoBoloMag'] - panDF['Mag']
    
    panDF['objInfoFlag'] = panDF['objInfoFlag'].apply(lambda x: f'0x{x:08X}')
    panDF['qualityFlag'] = panDF['qualityFlag'].apply(lambda x: f'0x{x:08X}')

    panDF, band1a, band1b, band2a, band2b, band1name, band2name = ColorCut(panDF, 'gr', 'ri')
    if (kronCutTF == True):
        panDF, kronName, psfName, kronCutName = KronCut(kronBand='g', kronDist = 0.5, dataDF = panDF)
    
    panDF2 = panDF
    #panDF2 = panDF[panDF["ginfoFlag2_Description"].apply(lambda x: flag_is_true(x, "SATSTAR_PROFILE"))]
    #print(len(panDF2))
    params = np.polyfit(panDF2['Mag'], panDF2['pseudoBoloMag'], 1, full=False, cov=True)

    slopeList.append(params[0][0])
    ABoffsetList.append(params[0][1])
    countsList.append(len(panDF2))
    exposures.append(n)

In [ ]:
dataTable = Table([julian_dates, slopeList, ABoffsetList, countsList, limiting_mags], names=['JulianDate', 'Slope', 'ABoffset', 'counts', 'limitingMag'])
# Creating the dataframe
data = {
    'JulianDate': julian_dates,
    'Slope': slopeList,
    'ABoffset': ABoffsetList,
    'counts': countsList,
    'limitingMag': limiting_mags
}

dataDF = pd.DataFrame(data)

In [ ]:
print(dataTable)
dataDF

In [ ]:
plt.figure(figsize=(18,9))
sns.scatterplot(x=julian_dates, y=slopeList)
plt.xlabel('Julian Date', fontsize=20)
plt.ylabel('Slope', fontsize=20)
plt.ylim(0.5, 1.0)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title('Slope of each exposure -- xtetrans_1a 0416 -- gKron 0.5', fontsize=20)
plt.show()

In [ ]:
plt.figure(figsize=(18,9))
sns.scatterplot(x=julian_dates, y=ABoffsetList)
plt.xlabel('Julian Date', fontsize=20)
plt.ylabel('AB Offset', fontsize=20)
plt.ylim(0.5, 4.5)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title('AB Offset of each exposure -- xtetrans_1a 0416 -- gKron 0.5', fontsize=20)
plt.show()

In [ ]:
plt.figure(figsize=(18,9))
sns.scatterplot(x=julian_dates, y=countsList)
plt.xlabel('Julian Date', fontsize=20)
plt.ylabel('# Matched to PanSTARRS', fontsize=20)
plt.ylim(0, 7000)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title('Number of objects matched to PanSTARRS of each exposure -- xtetrans_1a 0416 -- gKron 0.5', fontsize=20)
plt.show()

In [ ]:
plt.figure(figsize=(18,9))
sns.scatterplot(x=julian_dates, y=limiting_mags)
plt.xlabel('Julian Date', fontsize=20)
plt.ylabel('Limiting Magnitude', fontsize=20)
plt.ylim(12, 25)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.title('Limiting Magnitude of each exposure -- xtetrans_1a 0415 -- gKron 0.5', fontsize=20)
plt.show()

In [ ]:
corrM = dataDF.corr()
 
corrM

In [ ]:
plt.figure(figsize=(18,9))
ax = sns.heatmap(dataDF.corr(), annot=True)
plt.show()

In [ ]:
# Plotting
plt.figure(figsize=(14, 10))

# Slope vs counts
plt.subplot(2, 2, 1)
sns.scatterplot(x='Slope', y='counts', data=dataDF)
plt.title('Slope vs Counts')
plt.xlabel('Slope')
plt.ylabel('Counts')
plt.ylim(0, 7000)
plt.xlim(0.5, 1.0)

# ABoffset vs counts
plt.subplot(2, 2, 2)
sns.scatterplot(x='ABoffset', y='counts', data=dataDF)
plt.title('ABoffset vs Counts')
plt.xlabel('ABoffset')
plt.ylabel('Counts')
plt.ylim(0, 7000)
plt.xlim(0.5, 4.5)

# ABoffset vs Slope
plt.subplot(2, 2, 3)
sns.scatterplot(x='ABoffset', y='Slope', data=dataDF)
plt.title('ABoffset vs Slope')
plt.xlabel('ABoffset')
plt.ylabel('Slope')
plt.ylim(0.5, 1.0)
plt.xlim(0.5, 4.5)

# Counts vs Limiting Mag
plt.subplot(2, 2, 4)
sns.scatterplot(x='counts', y='limitingMag', data=dataDF)
plt.title('Counts vs Limiting Magnitude')
plt.xlabel('Counts')
plt.ylabel('Limiting Magnitude')
plt.ylim(12, 25)
plt.xlim(0, 7000)

plt.tight_layout()
plt.savefig("xet_1a0416.png")
plt.show()

In [ ]:
# Histogram plots
plt.figure(figsize=(18, 12))

# Histogram for Slope
plt.subplot(2, 2, 1)
sns.histplot(data=dataDF, x='Slope', bins=30)
plt.title('Histogram of Slope')
plt.xlabel('Slope')
plt.ylabel('Count')

# Histogram for ABoffset
plt.subplot(2, 2, 2)
sns.histplot(data=dataDF, x='ABoffset', bins=30)
plt.title('Histogram of ABoffset')
plt.xlabel('ABoffset')
plt.ylabel('Count')

# Histogram for Counts
plt.subplot(2, 2, 3)
sns.histplot(data=dataDF, x='counts', bins=30)
plt.title('Histogram of Counts')
plt.xlabel('Counts')
plt.ylabel('Count')

# Histogram for Limiting Magnitude
plt.subplot(2, 2, 4)
sns.histplot(data=dataDF, x='limitingMag', bins=30)
plt.title('Histogram of Limiting Magnitude')
plt.xlabel('Limiting Magnitude')
plt.ylabel('Count')

plt.tight_layout()
plt.show()